In [1]:
import numpy as np
from chaotic_sim import (
    BENCHMARK_SUITE,
    ChaoticSimulatedAnnealing,
    PRNGGenerator,
    LogisticMapGenerator,
    TentMapGenerator,
    HenonMapGenerator,
)

print("=== SMOKE-TEST: Валидация работы модуля chaotic_sim.py ===")

# Тестовая функция Растригина
rastrigin_bench = BENCHMARK_SUITE["Rastrigin"]
dim = 5
n_iters = 1000

# Семейство генераторов
generators = {
    "PRNG (Uniform)": PRNGGenerator(seed=42),
    "Logistic Map": LogisticMapGenerator(seed=42),
    "Tent Map": TentMapGenerator(seed=42),
    "Henon Map": HenonMapGenerator(seed=42),
}

for name, gen in generators.items():
    optimizer = ChaoticSimulatedAnnealing(
        benchmark=rastrigin_bench,
        dimension=dim,
        generator=gen,
        t_init=15.0,
        t_final=1e-4,
        cooling_factor=0.992,
        alpha_init=0.25,
        alpha_decay=0.995,
        max_iterations=n_iters,
        epsilon=1e-2,
        seed=100,
    )
    res = optimizer.optimize()
    fht_str = str(res.fht) if res.fht is not None else "Не достигнут"
    print(f"[{name:<15}] Рекордная энергия: {res.best_energy:10.5f} | FHT (eps=0.01): {fht_str:<12} [УСПЕХ]")

print("\nВсе модули ядра успешно прошли тестирование и готовы к масштабным статистическим экспериментам!")

=== SMOKE-TEST: Валидация работы модуля chaotic_sim.py ===
[PRNG (Uniform) ] Рекордная энергия:    8.96848 | FHT (eps=0.01): Не достигнут [УСПЕХ]
[Logistic Map   ] Рекордная энергия:   10.01563 | FHT (eps=0.01): Не достигнут [УСПЕХ]
[Tent Map       ] Рекордная энергия:   31.48220 | FHT (eps=0.01): Не достигнут [УСПЕХ]
[Henon Map      ] Рекордная энергия:   51.79699 | FHT (eps=0.01): Не достигнут [УСПЕХ]

Все модули ядра успешно прошли тестирование и готовы к масштабным статистическим экспериментам!


In [3]:
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu
from chaotic_sim import (
    BENCHMARK_SUITE,
    ChaoticSimulatedAnnealing,
    PRNGGenerator,
    LogisticMapGenerator,
    TentMapGenerator,
    HenonMapGenerator,
)

# -------------------------------------------------------------
# 1. КОНФИГУРАЦИЯ СТАТИСТИЧЕСКОГО ЭКСПЕРИМЕНТА
# -------------------------------------------------------------
N_RUNS = 50          # Число независимых запусков
DIM = 5              # Размерность задачи d = 5
MAX_ITERS = 1500     # Длина траектории отжига
EPSILON = 1e-2       # Порог окрестности оптимума

GENERATOR_CLASSES = {
    "PRNG (Uniform)": PRNGGenerator,
    "Logistic Map": LogisticMapGenerator,
    "Tent Map": TentMapGenerator,
    "Henon Map": HenonMapGenerator,
}

BENCHMARKS_TO_TEST = ["Sphere", "Rastrigin", "Griewank", "ShiftedSphere"]

results_data = []
convergence_histories = {b_name: {g_name: [] for g_name in GENERATOR_CLASSES} for b_name in BENCHMARKS_TO_TEST}

print(f"Запуск статистического эксперимента: {len(BENCHMARKS_TO_TEST)} функций x {len(GENERATOR_CLASSES)} генераторов x {N_RUNS} запусков...")

# -------------------------------------------------------------
# 2. ОСНОВНОЙ ВЫЧИСЛИТЕЛЬНЫЙ ЦИКЛ
# -------------------------------------------------------------
for b_name in BENCHMARKS_TO_TEST:
    bench = BENCHMARK_SUITE[b_name]
    print(f"\nТестирование функции: {b_name} (bounds={bench.bounds}, center={bench.center_optimum})")

    bench_runs = {}

    for g_name, g_cls in GENERATOR_CLASSES.items():
        energies = []
        fhts = []

        for run_idx in range(N_RUNS):
            seed = 1000 * (run_idx + 1) + 42
            gen = g_cls(seed=seed)

            optimizer = ChaoticSimulatedAnnealing(
                benchmark=bench,
                dimension=DIM,
                generator=gen,
                t_init=15.0,
                t_final=1e-4,
                cooling_factor=0.993,
                alpha_init=0.25,
                alpha_decay=0.997,
                max_iterations=MAX_ITERS,
                epsilon=EPSILON,
                seed=seed,
            )
            res = optimizer.optimize()
            energies.append(res.best_energy)
            fhts.append(res.fht if res.fht is not None else MAX_ITERS)
            convergence_histories[b_name][g_name].append(res.energy_history)

        bench_runs[g_name] = np.array(energies)

    # ---------------------------------------------------------
    # 3. РАСЧЕТ СТАТИСТИК И КРИТЕРИЯ МАННА-УИТНИ
    # ---------------------------------------------------------
    baseline_energies = bench_runs["PRNG (Uniform)"]

    for g_name in GENERATOR_CLASSES:
        vals = bench_runs[g_name]
        median_val = np.median(vals)
        q25 = np.percentile(vals, 25)
        q75 = np.percentile(vals, 75)
        iqr_val = q75 - q25

        if g_name == "PRNG (Uniform)":
            p_str = "---"
            p_val = 1.0
        else:
            stat, p_val = mannwhitneyu(vals, baseline_energies, alternative='two-sided')
            p_str = f"{p_val:.4f}" if p_val >= 0.0001 else "< 0.0001"

        success_rate = np.mean(vals <= EPSILON) * 100.0

        results_data.append({
            "Функция": b_name,
            "Генератор": g_name,
            "Медиана E": median_val,
            "IQR": iqr_val,
            "Success (%)": success_rate,
            "p-value": p_str,
        })
        print(f"  [{g_name:<15}] Медиана: {median_val:10.4e} | IQR: {iqr_val:10.4e} | p-value: {p_str}")

df_results = pd.DataFrame(results_data)

# Отображение таблицы в самом блокноте (без сохранения в .tex)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
display(df_results)

Запуск статистического эксперимента: 4 функций x 4 генераторов x 50 запусков...

Тестирование функции: Sphere (bounds=(-5.12, 5.12), center=True)
  [PRNG (Uniform) ] Медиана: 3.1847e-04 | IQR: 1.4297e-04 | p-value: ---
  [Logistic Map   ] Медиана: 5.7967e-04 | IQR: 3.6273e-04 | p-value: < 0.0001
  [Tent Map       ] Медиана: 6.5907e-04 | IQR: 4.2447e-04 | p-value: < 0.0001
  [Henon Map      ] Медиана: 1.6256e-01 | IQR: 5.8166e-01 | p-value: < 0.0001

Тестирование функции: Rastrigin (bounds=(-5.12, 5.12), center=True)
  [PRNG (Uniform) ] Медиана: 1.0993e+01 | IQR: 8.0007e+00 | p-value: ---
  [Logistic Map   ] Медиана: 1.0119e+01 | IQR: 9.6799e+00 | p-value: 0.9972
  [Tent Map       ] Медиана: 3.9064e+01 | IQR: 1.5080e+01 | p-value: < 0.0001
  [Henon Map      ] Медиана: 4.6494e+01 | IQR: 1.5816e+01 | p-value: < 0.0001

Тестирование функции: Griewank (bounds=(-10.0, 10.0), center=True)
  [PRNG (Uniform) ] Медиана: 4.5859e-02 | IQR: 2.2153e-02 | p-value: ---
  [Logistic Map   ] Медиана: 3.6

,Функция,Генератор,Медиана E,IQR,Success (%),p-value
0,Sphere,PRNG (Uniform),0.000318,0.000143,100.0,---
1,Sphere,Logistic Map,0.000580,0.000363,100.0,< 0.0001
2,Sphere,Tent Map,0.000659,0.000424,100.0,< 0.0001
3,Sphere,Henon Map,0.162557,0.581661,4.0,< 0.0001
4,Rastrigin,PRNG (Uniform),10.992803,8.000667,0.0,---
5,Rastrigin,Logistic Map,10.118806,9.679883,0.0,0.9972
6,Rastrigin,Tent Map,39.063927,15.080201,0.0,< 0.0001
7,Rastrigin,Henon Map,46.493834,15.816384,0.0,< 0.0001
8,Griewank,PRNG (Uniform),0.045859,0.022153,0.0,---
9,Griewank,Logistic Map,0.036244,0.022547,2.0,0.6969


In [4]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('default')
plt.rcParams.update({
    'font.size': 11,
    'font.family': 'serif',
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'savefig.facecolor': 'white',
    'text.usetex': False
})

ru_generator_names = {
    "PRNG (Uniform)": "БСВ (равномерный)",
    "Logistic Map": "Логистическое",
    "Tent Map": "Палатка",
    "Henon Map": "Энон",
}

benchmarks_meta = {
    "Sphere": {
        "title": "Сферическая функция ($d=5$)",
        "filename": "conv_sphere.pdf"
    },
    "Rastrigin": {
        "title": "Функция Растригина ($d=5$)",
        "filename": "conv_rastrigin.pdf"
    },
    "Griewank": {
        "title": "Функция Гриванка ($d=5$)",
        "filename": "conv_griewank.pdf"
    },
    "ShiftedSphere": {
        "title": "Смещённая сфера ($d=5$)",
        "filename": "conv_shiftedsphere.pdf"
    }
}

colors = {
    "PRNG (Uniform)": "black",
    "Logistic Map": "crimson",
    "Tent Map": "forestgreen",
    "Henon Map": "indigo",
}
styles = {
    "PRNG (Uniform)": "--",
    "Logistic Map": "-",
    "Tent Map": "-",
    "Henon Map": "-.",
}

# Экспорт каждого графика в отдельный векторный файл PDF
for b_name, meta in benchmarks_meta.items():
    fig, ax = plt.subplots(figsize=(6, 4.2), dpi=300, facecolor='white')

    for g_name in GENERATOR_CLASSES:
        hist_matrix = np.array(convergence_histories[b_name][g_name])
        median_curve = np.median(hist_matrix, axis=0)
        ax.plot(
            median_curve,
            label=ru_generator_names[g_name],
            color=colors[g_name],
            linestyle=styles[g_name],
            lw=1.8
        )

    ax.set_title(meta["title"])
    ax.set_xlabel("Итерация")
    ax.set_ylabel("Медианная энергия")
    ax.set_yscale("log")
    ax.grid(True, which="both", ls=":", lw=0.5, alpha=0.7)
    ax.legend(frameon=True, facecolor='white', edgecolor='none')

    plt.tight_layout()
    plt.savefig(meta["filename"], facecolor='white', transparent=False)
    plt.close()
    print(f"Файл сохранен: {meta['filename']}")

print("\nВсе 4 графика сходимости сохранены по отдельности!")

Файл сохранен: conv_sphere.pdf
Файл сохранен: conv_rastrigin.pdf
Файл сохранен: conv_griewank.pdf
Файл сохранен: conv_shiftedsphere.pdf

Все 4 графика сходимости сохранены по отдельности!
